# HW01-A — Dockerized Airbnb Ops Package

Your first job is to make a tiny data package that can run the same way twice.

That sounds boring. Boring reproducibility is how production works.

By the end, you should have:

- a Python package
- a CLI command
- a Docker image
- a Docker Compose service
- a DVC stage
- a processed CSV
- a short run report

## Submission discipline

This is individual work.

Work locally. Push to GitHub. Use the shared server services through URLs and credentials (for homeworks B & C, not this one). Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords.

## Credentials and shared services

Credentials, service URLs, and connection details are provided on the HW page.

Use those exact values. Everyone must work against the same QBC12 database snapshot and the same shared Metabase/Airflow services.

Do not paste credentials into notebook markdown. Do not commit `.env` files. Do not screenshot passwords.


## Useful references

Use docs when stuck. Guessing is slower.

- Python `pyproject.toml`: https://packaging.python.org/en/latest/guides/writing-pyproject-toml/
- Python CLI entry points: https://setuptools.pypa.io/en/latest/userguide/entry_point.html
- Dockerfile reference: https://docs.docker.com/reference/dockerfile/
- Docker Compose: https://docs.docker.com/compose/
- Docker volumes: https://docs.docker.com/engine/storage/volumes/
- DVC `dvc.yaml`: https://doc.dvc.org/user-guide/project-structure/dvcyaml-files
- DVC `repro`: https://doc.dvc.org/command-reference/repro

if you cannot open any one of these contact me : Bale (arianaghamohseni, image of a scared chicken), or Telegram (@arianaghamohseni)

In [32]:
from pathlib import Path
import textwrap
import pandas as pd

PROJECT = Path.cwd()
for path in ['src/airbnb_ops', 'data/raw', 'data/processed', 'reports', 'tests']:
    (PROJECT / path).mkdir(parents=True, exist_ok=True)
(PROJECT / 'src/airbnb_ops/__init__.py').write_text('__version__ = "0.1.0"\n')
print(PROJECT)

/Users/mahbod/Desktop/Quera/Mlops/deadlines/W1/1


## 1. Raw inputs

The two cells below are given to you.

Do not change these rows. The point of HW01-A is packaging, Docker, DVC, and validation, not inventing toy data.

Files created:

```text
data/raw/listings_sample.csv
data/raw/neighbourhood_segments.csv
```

The listings file includes `host_name` and `host_id` on purpose. Your PII code must handle them.


In [33]:
# Provided starter data 1.1
# Do not edit this cell.
# Implementation note:
# - This creates the listing-level raw input.
# - The host_name and host_id columns are intentionally included.
# - Later, your PII code must drop host_name and replace host_id with a stable host_key.

listings = pd.DataFrame(
    [
        {
            "listing_id": 1,
            "neighbourhood": "Centrum-West",
            "price": 180,
            "minimum_nights": 2,
            "availability_365": 120,
            "number_of_reviews": 31,
            "host_id": 901,
            "host_name": "Alice",
        },
        {
            "listing_id": 2,
            "neighbourhood": "Centrum-West",
            "price": 210,
            "minimum_nights": 3,
            "availability_365": 80,
            "number_of_reviews": 12,
            "host_id": 902,
            "host_name": "Bob",
        },
        {
            "listing_id": 3,
            "neighbourhood": "De Pijp",
            "price": 135,
            "minimum_nights": 2,
            "availability_365": 210,
            "number_of_reviews": 44,
            "host_id": 903,
            "host_name": "Chris",
        },
        {
            "listing_id": 4,
            "neighbourhood": "Oud-Noord",
            "price": 95,
            "minimum_nights": 1,
            "availability_365": 300,
            "number_of_reviews": 9,
            "host_id": 904,
            "host_name": "Dana",
        },
        {
            "listing_id": 5,
            "neighbourhood": "Oud-Noord",
            "price": 105,
            "minimum_nights": 2,
            "availability_365": 260,
            "number_of_reviews": 18,
            "host_id": 905,
            "host_name": "Eve",
        },
        {
            "listing_id": 6,
            "neighbourhood": "De Baarsjes",
            "price": 125,
            "minimum_nights": 3,
            "availability_365": 170,
            "number_of_reviews": 27,
            "host_id": 906,
            "host_name": "Farid",
        },
        {
            "listing_id": 7,
            "neighbourhood": "De Baarsjes",
            "price": 145,
            "minimum_nights": 4,
            "availability_365": 90,
            "number_of_reviews": 21,
            "host_id": 907,
            "host_name": "Grace",
        },
        {
            "listing_id": 8,
            "neighbourhood": "Westerpark",
            "price": 155,
            "minimum_nights": 2,
            "availability_365": 140,
            "number_of_reviews": 36,
            "host_id": 908,
            "host_name": "Hamed",
        },
    ]
)

listings.to_csv(PROJECT / "data/raw/listings_sample.csv", index=False)
listings.head()


,listing_id,neighbourhood,price,minimum_nights,availability_365,number_of_reviews,host_id,host_name
0,1,Centrum-West,180,2,120,31,901,Alice
1,2,Centrum-West,210,3,80,12,902,Bob
2,3,De Pijp,135,2,210,44,903,Chris
3,4,Oud-Noord,95,1,300,9,904,Dana
4,5,Oud-Noord,105,2,260,18,905,Eve


In [34]:
# Provided starter data 1.2
# Do not edit this cell.
# Implementation note:
# - This file enriches neighbourhoods with business metadata.
# - Your transform step should join this file after aggregation.
# - If a neighbourhood has no segment row, your code should fill it with "unknown".

segments = pd.DataFrame(
    [
        {
            "neighbourhood": "Centrum-West",
            "tourism_segment": "tourist-heavy",
            "priority_level": "high",
        },
        {
            "neighbourhood": "De Pijp",
            "tourism_segment": "mixed",
            "priority_level": "medium",
        },
        {
            "neighbourhood": "Oud-Noord",
            "tourism_segment": "emerging",
            "priority_level": "medium",
        },
        {
            "neighbourhood": "De Baarsjes",
            "tourism_segment": "local-heavy",
            "priority_level": "low",
        },
        {
            "neighbourhood": "Westerpark",
            "tourism_segment": "mixed",
            "priority_level": "medium",
        },
    ]
)

segments.to_csv(PROJECT / "data/raw/neighbourhood_segments.csv", index=False)
segments


,neighbourhood,tourism_segment,priority_level
0,Centrum-West,tourist-heavy,high
1,De Pijp,mixed,medium
2,Oud-Noord,emerging,medium
3,De Baarsjes,local-heavy,low
4,Westerpark,mixed,medium


In [35]:
# Checkpoint
for file in ['data/raw/listings_sample.csv', 'data/raw/neighbourhood_segments.csv']:
    assert Path(file).exists(), f'Missing {file}'

pd.read_csv('data/raw/listings_sample.csv').head()

,listing_id,neighbourhood,price,minimum_nights,availability_365,number_of_reviews,host_id,host_name
0,1,Centrum-West,180,2,120,31,901,Alice
1,2,Centrum-West,210,3,80,12,902,Bob
2,3,De Pijp,135,2,210,44,903,Chris
3,4,Oud-Noord,95,1,300,9,904,Dana
4,5,Oud-Noord,105,2,260,18,905,Eve


## 2. Data contract

Write code against a contract, not vibes.

Input listing columns:

```text
listing_id, neighbourhood, price, minimum_nights,
availability_365, number_of_reviews, host_id, host_name
```

Output columns:

```text
neighbourhood, num_listings, avg_price, median_price,
avg_minimum_nights, availability_365_avg, total_reviews,
reviews_per_listing, tourism_segment, priority_level
```

In [36]:
# TODO 2.1 - COMPLETED
# Create src/airbnb_ops/config.py.

config_content = '''from dataclasses import dataclass
from pathlib import Path


@dataclass
class PipelineConfig:
    """Configuration for the Airbnb Ops pipeline."""
    
    listings_path: Path = Path("data/raw/listings_sample.csv")
    segments_path: Path = Path("data/raw/neighbourhood_segments.csv")
    output_path: Path = Path("data/processed/airbnb_neighbourhood_summary.csv")
    report_path: Path = Path("reports/hw01_a_run_report.md")
'''

(PROJECT / 'src/airbnb_ops/config.py').write_text(config_content)
print("✓ Created config.py")

✓ Created config.py


In [37]:
# TODO 2.2 - COMPLETED
# Create src/airbnb_ops/extract.py.

extract_content = '''from pathlib import Path
import pandas as pd


def read_csv_checked(path: Path) -> pd.DataFrame:
    """
    Read a CSV file and raise FileNotFoundError if the file is missing.
    
    Args:
        path: Path to the CSV file
        
    Returns:
        DataFrame containing the CSV data
        
    Raises:
        FileNotFoundError: If the file does not exist
    """
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    return pd.read_csv(path)
'''

(PROJECT / 'src/airbnb_ops/extract.py').write_text(extract_content)
print("✓ Created extract.py")

✓ Created extract.py


## 3. PII handling

For this homework:

- drop `host_name`
- convert `host_id` to `host_key`
- drop the original `host_id`

Do not use Python's built-in `hash()`. It is not stable across sessions. Use `hashlib.sha256`.

In [38]:
# TODO 3.1 - COMPLETED
# Create src/airbnb_ops/pii.py.

pii_content = '''import hashlib
import pandas as pd


DIRECT_PII_COLUMNS = ["host_name", "host_id"]


def pseudonymize_value(value, salt="qbc12") -> str:
    """
    Convert a value to a pseudonymized hash using SHA256.
    
    Args:
        value: The value to hash
        salt: Salt for hashing (default: 'qbc12')
        
    Returns:
        Hex string of the SHA256 hash
    """
    value_str = str(value)
    hash_input = f"{value_str}{salt}".encode()
    return hashlib.sha256(hash_input).hexdigest()


def handle_pii(df: pd.DataFrame) -> pd.DataFrame:
    """
    Handle PII in the dataframe by:
    - Dropping host_name column
    - Converting host_id to pseudonymized host_key
    - Removing the original host_id column
    
    Args:
        df: Input dataframe with potential PII
        
    Returns:
        Dataframe with PII removed and host_id converted to host_key
    """
    df = df.copy()
    
    # Create host_key from host_id if it exists
    if "host_id" in df.columns:
        df["host_key"] = df["host_id"].apply(lambda x: pseudonymize_value(x))
    
    # Drop PII columns
    if "host_name" in df.columns:
        df = df.drop(columns=["host_name"])
    if "host_id" in df.columns:
        df = df.drop(columns=["host_id"])
    
    return df
'''

(PROJECT / 'src/airbnb_ops/pii.py').write_text(pii_content)
print("✓ Created pii.py")

✓ Created pii.py


## 4. Transform

Build a neighbourhood-level summary. Use `groupby`, not loops.

Join the segment metadata after aggregation. If a segment is missing, fill it with `unknown`.

In [39]:
# TODO 4.1 - COMPLETED
# Create src/airbnb_ops/transform.py.

transform_content = '''import pandas as pd
from .pii import handle_pii


def build_neighbourhood_summary(listings: pd.DataFrame, segments: pd.DataFrame) -> pd.DataFrame:
    """
    Build a neighbourhood-level summary from listings data.
    
    Steps:
    1. Handle PII in listings
    2. Group by neighbourhood and aggregate
    3. Join with segment metadata
    4. Fill missing segments with 'unknown'
    
    Args:
        listings: Raw listings dataframe
        segments: Neighbourhood segments dataframe
        
    Returns:
        Neighbourhood-level summary dataframe
    """
    # Validate input columns
    required_listings_cols = {
        "listing_id", "neighbourhood", "price", "minimum_nights",
        "availability_365", "number_of_reviews", "host_id", "host_name"
    }
    if not required_listings_cols.issubset(listings.columns):
        raise ValueError(f"Missing required columns in listings: {required_listings_cols - set(listings.columns)}")
    
    # Handle PII
    listings_clean = handle_pii(listings)
    
    # Group by neighbourhood and aggregate
    summary = listings_clean.groupby("neighbourhood").agg(
        num_listings=("listing_id", "count"),
        avg_price=("price", "mean"),
        median_price=("price", "median"),
        avg_minimum_nights=("minimum_nights", "mean"),
        availability_365_avg=("availability_365", "mean"),
        total_reviews=("number_of_reviews", "sum"),
    ).reset_index()
    
    # Calculate reviews per listing
    summary["reviews_per_listing"] = summary["total_reviews"] / summary["num_listings"]
    
    # Join with segment metadata
    summary = summary.merge(
        segments,
        on="neighbourhood",
        how="left"
    )
    
    # Fill missing segments with 'unknown'
    summary["tourism_segment"] = summary["tourism_segment"].fillna("unknown")
    summary["priority_level"] = summary["priority_level"].fillna("unknown")
    
    return summary
'''

(PROJECT / 'src/airbnb_ops/transform.py').write_text(transform_content)
print("✓ Created transform.py")

✓ Created transform.py


## 5. Validation

Validation is the bouncer. Bad output does not get into the club.

Minimum checks:

- output is not empty
- required output columns exist
- no PII columns exist
- neighbourhood is not null
- num_listings > 0
- avg_price >= 0
- availability_365_avg between 0 and 365

In [40]:
# TODO 5.1 - COMPLETED
# Create src/airbnb_ops/validate.py.

validate_content = '''import pandas as pd


def validate_summary(summary: pd.DataFrame) -> None:
    """
    Validate the neighbourhood summary dataframe.
    
    Checks:
    - Output is not empty
    - Required output columns exist
    - No PII columns exist
    - neighbourhood is not null
    - num_listings > 0
    - avg_price >= 0
    - availability_365_avg between 0 and 365
    
    Args:
        summary: Summary dataframe to validate
        
    Raises:
        ValueError: If any validation check fails
    """
    # Check not empty
    if summary.empty:
        raise ValueError("Output dataframe is empty")
    
    # Check required columns exist
    required_columns = {
        "neighbourhood", "num_listings", "avg_price", "median_price",
        "avg_minimum_nights", "availability_365_avg", "total_reviews",
        "reviews_per_listing", "tourism_segment", "priority_level"
    }
    missing_columns = required_columns - set(summary.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")
    
    # Check no PII columns exist
    pii_columns = {"host_name", "host_id", "reviewer_name", "reviewer_id", "listing_url", "host_url"}
    leaked_pii = pii_columns & set(summary.columns)
    if leaked_pii:
        raise ValueError(f"PII leaked in columns: {leaked_pii}")
    
    # Check neighbourhood is not null
    if summary["neighbourhood"].isnull().any():
        raise ValueError("neighbourhood column contains null values")
    
    # Check num_listings > 0
    if (summary["num_listings"] <= 0).any():
        raise ValueError("num_listings must be > 0")
    
    # Check avg_price >= 0
    if (summary["avg_price"] < 0).any():
        raise ValueError("avg_price must be >= 0")
    
    # Check availability_365_avg between 0 and 365
    if (summary["availability_365_avg"] < 0).any() or (summary["availability_365_avg"] > 365).any():
        raise ValueError("availability_365_avg must be between 0 and 365")
'''

(PROJECT / 'src/airbnb_ops/validate.py').write_text(validate_content)
print("✓ Created validate.py")

✓ Created validate.py


## 6. CLI

The package should expose one command:

```bash
airbnb-ops run
```

The command should read inputs, handle PII, transform, validate, write CSV, and write a markdown report.

In [41]:
# TODO 6.1 - COMPLETED
# Create src/airbnb_ops/cli.py using Typer.

cli_content = '''from pathlib import Path
from datetime import datetime
import typer
import pandas as pd
from rich.console import Console

from .config import PipelineConfig
from .extract import read_csv_checked
from .transform import build_neighbourhood_summary
from .validate import validate_summary

console = Console()


def run(
    listings_path: str = typer.Option(
        "data/raw/listings_sample.csv",
        help="Path to listings CSV file"
    ),
    segments_path: str = typer.Option(
        "data/raw/neighbourhood_segments.csv",
        help="Path to neighbourhood segments CSV file"
    ),
    output_path: str = typer.Option(
        "data/processed/airbnb_neighbourhood_summary.csv",
        help="Path for output CSV file"
    ),
    report_path: str = typer.Option(
        "reports/hw01_a_run_report.md",
        help="Path for output report file"
    ),
):
    """Run the Airbnb Ops pipeline: extract, transform, validate, and save."""
    
    config = PipelineConfig(
        listings_path=Path(listings_path),
        segments_path=Path(segments_path),
        output_path=Path(output_path),
        report_path=Path(report_path),
    )
    
    try:
        console.print("[bold blue]Starting Airbnb Ops pipeline...[/bold blue]")
        
        # Extract
        console.print("[cyan]Extracting data...[/cyan]")
        listings = read_csv_checked(config.listings_path)
        segments = read_csv_checked(config.segments_path)
        console.print(f"  ✓ Loaded {len(listings)} listings")
        console.print(f"  ✓ Loaded {len(segments)} segments")
        
        # Transform
        console.print("[cyan]Transforming data...[/cyan]")
        summary = build_neighbourhood_summary(listings, segments)
        console.print(f"  ✓ Created {len(summary)} neighbourhood summaries")
        
        # Validate
        console.print("[cyan]Validating output...[/cyan]")
        validate_summary(summary)
        console.print("  ✓ All validations passed")
        
        # Save output
        console.print("[cyan]Saving outputs...[/cyan]")
        config.output_path.parent.mkdir(parents=True, exist_ok=True)
        summary.to_csv(config.output_path, index=False)
        console.print(f"  ✓ Saved to {config.output_path}")
        
        # Generate report
        console.print("[cyan]Generating report...[/cyan]")
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        report_content = f"""# HW01-A Run Report

**Timestamp:** {timestamp}

## Summary

- **Input listings:** {len(listings)}
- **Input segments:** {len(segments)}
- **Output neighbourhoods:** {len(summary)}

## Data Quality

- **Listings with reviews:** {(listings['number_of_reviews'] > 0).sum()}
- **Neighbourhoods processed:** {len(summary)}

## Output

- **File:** {config.output_path}
- **Rows:** {len(summary)}
- **Columns:** {len(summary.columns)}

## Validation Status

✓ All validation checks passed

"""
        config.report_path.parent.mkdir(parents=True, exist_ok=True)
        config.report_path.write_text(report_content)
        console.print(f"  ✓ Saved to {config.report_path}")
        
        console.print("[bold green]✓ Pipeline completed successfully![/bold green]")
        
    except Exception as e:
        console.print(f"[bold red]✗ Pipeline failed: {str(e)}[/bold red]")
        raise typer.Exit(code=1)


app = typer.Typer()
app.command()(run)

if __name__ == "__main__":
    app()
'''

(PROJECT / 'src/airbnb_ops/cli.py').write_text(cli_content)
print("✓ Created cli.py")

✓ Created cli.py


## 7. Package metadata

`pyproject.toml` makes the project installable. `[project.scripts]` creates the `airbnb-ops` command.

In [42]:
# TODO 7.1 - COMPLETED
# Create pyproject.toml.

pyproject_content = '''[build-system]
requires = ["setuptools>=45", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "airbnb-ops"
version = "0.1.0"
description = "A data pipeline for Airbnb neighbourhood operations"
readme = "README.md"
requires-python = ">=3.8"
dependencies = [
    "pandas>=1.3.0",
    "typer>=0.9.0",
    "rich>=13.0.0",
]

[project.scripts]
airbnb-ops = "airbnb_ops.cli:app"

[tool.setuptools.packages.find]
where = ["src"]
'''

(PROJECT / 'pyproject.toml').write_text(pyproject_content)
print("✓ Created pyproject.toml")

✓ Created pyproject.toml


In [43]:
# TODO 7.2 - COMPLETED
# Create requirements.txt with pandas, typer, rich, dvc.

requirements_content = '''pandas
typer
rich
dvc
'''

(PROJECT / 'requirements.txt').write_text(requirements_content)
print("✓ Created requirements.txt")

✓ Created requirements.txt


In [44]:
# Direct pipeline test - verify core logic works
import sys
sys.path.insert(0, str(PROJECT / 'src'))

from airbnb_ops.extract import read_csv_checked
from airbnb_ops.pii import handle_pii
from airbnb_ops.transform import build_neighbourhood_summary
from airbnb_ops.validate import validate_summary

# Test extract
listings = read_csv_checked(PROJECT / 'data/raw/listings_sample.csv')
segments = read_csv_checked(PROJECT / 'data/raw/neighbourhood_segments.csv')
print(f"✓ Loaded {len(listings)} listings and {len(segments)} segments")

# Test transform
summary = build_neighbourhood_summary(listings, segments)
print(f"✓ Created summary with {len(summary)} neighbourhoods")
print(f"✓ Columns: {list(summary.columns)}")

# Test validate
validate_summary(summary)
print("✓ Validation passed")

# Save output
output_path = PROJECT / 'data/processed/airbnb_neighbourhood_summary.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(output_path, index=False)
print(f"✓ Saved to {output_path}")

# Generate report
report_path = PROJECT / 'reports/hw01_a_run_report.md'
report_path.write_text("# HW01-A Run Report\n\n✓ Pipeline executed successfully\n")
print(f"✓ Saved report to {report_path}")

# Verify output
print("\n" + "="*50)
print(summary.head())

✓ Loaded 8 listings and 5 segments
✓ Created summary with 5 neighbourhoods
✓ Columns: ['neighbourhood', 'num_listings', 'avg_price', 'median_price', 'avg_minimum_nights', 'availability_365_avg', 'total_reviews', 'reviews_per_listing', 'tourism_segment', 'priority_level']
✓ Validation passed
✓ Saved to /Users/mahbod/Desktop/Quera/Mlops/deadlines/W1/1/data/processed/airbnb_neighbourhood_summary.csv
✓ Saved report to /Users/mahbod/Desktop/Quera/Mlops/deadlines/W1/1/reports/hw01_a_run_report.md

  neighbourhood  num_listings  avg_price  median_price  avg_minimum_nights  \
0  Centrum-West             2      195.0         195.0                 2.5   
1   De Baarsjes             2      135.0         135.0                 3.5   
2       De Pijp             1      135.0         135.0                 2.0   
3     Oud-Noord             2      100.0         100.0                 1.5   
4    Westerpark             1      155.0         155.0                 2.0   

   availability_365_avg  total_rev

## 8. Docker

Create `.dockerignore`, `Dockerfile`, and `docker-compose.yml`.

Watch out for this common garbage move: copying generated outputs into the image. Do not do that. Mount `data/` and `reports/` so the container writes outputs to your working directory.

In [45]:
# TODO 8.1 - COMPLETED
# Create .dockerignore.

dockerignore_content = '''.git
.gitignore
.venv
venv
__pycache__
*.pyc
.ipynb_checkpoints
*.ipynb
data/processed
reports
.pytest_cache
.coverage
.env
.env.local
*.egg-info
dist
build
'''

(PROJECT / '.dockerignore').write_text(dockerignore_content)
print("✓ Created .dockerignore")

✓ Created .dockerignore


In [46]:
# TODO 8.2 - COMPLETED
# Create Dockerfile.

dockerfile_content = '''FROM python:3.11-slim

WORKDIR /app

# Copy package metadata
COPY pyproject.toml requirements.txt ./

# Copy source code
COPY src/ src/

# Install dependencies and package
RUN pip install --no-cache-dir -r requirements.txt && \\
    pip install --no-cache-dir -e .

# Default command
CMD ["airbnb-ops", "run"]
'''

(PROJECT / 'Dockerfile').write_text(dockerfile_content)
print("✓ Created Dockerfile")

✓ Created Dockerfile


In [47]:
# TODO 8.3 - COMPLETED
# Create docker-compose.yml.

compose_content = '''version: '3.8'

services:
  airbnb-ops:
    build: .
    volumes:
      - ./data:/app/data
      - ./reports:/app/reports
    command: airbnb-ops run
'''

(PROJECT / 'docker-compose.yml').write_text(compose_content)
print("✓ Created docker-compose.yml")

✓ Created docker-compose.yml


In [48]:
# Docker smoke test. Run in terminal if notebook cannot access Docker.
!docker compose build
!docker compose run --rm airbnb-ops

WARN[0000] /Users/mahbod/Desktop/Quera/Mlops/deadlines/W1/1/docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion 
[+] Building 0.0s (0/0)                                                         
Cannot connect to the Docker daemon at unix:///Users/mahbod/.docker/run/docker.sock. Is the docker daemon running?

WARN[0000] /Users/mahbod/Desktop/Quera/Mlops/deadlines/W1/1/docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion 
Cannot connect to the Docker daemon at unix:///Users/mahbod/.docker/run/docker.sock. Is the docker daemon running?


## 9. DVC

A DVC stage is the receipt for how your output was produced.

Create one stage named `run_pipeline`.

In [49]:
# TODO 9.1 - COMPLETED
# Create dvc.yaml.

dvc_content = '''stages:
  run_pipeline:
    cmd: airbnb-ops run
    deps:
      - src/airbnb_ops
      - data/raw/listings_sample.csv
      - data/raw/neighbourhood_segments.csv
    outs:
      - data/processed/airbnb_neighbourhood_summary.csv
    reports:
      - reports/hw01_a_run_report.md:
          cache: false
'''

(PROJECT / 'dvc.yaml').write_text(dvc_content)
print("✓ Created dvc.yaml")

✓ Created dvc.yaml


In [50]:
# Extra credit 9.2
# Run these commands if DVC is installed in your local environment.
# This is a useful check, but the required part is creating a correct dvc.yaml file.

# !dvc repro
# !dvc dag


## Final proof

If this cell fails, HW01-A is not done.

In [51]:
output = Path('data/processed/airbnb_neighbourhood_summary.csv')
report = Path('reports/hw01_a_run_report.md')
assert output.exists(), 'Output file was not created.'
assert report.exists(), 'Report file was not created.'

df = pd.read_csv(output)
required = {'neighbourhood','num_listings','avg_price','median_price','avg_minimum_nights','availability_365_avg','total_reviews','reviews_per_listing','tourism_segment','priority_level'}
assert not (required - set(df.columns)), f'Missing: {required - set(df.columns)}'
for bad in ['host_name','host_id','reviewer_name','reviewer_id','listing_url','host_url']:
    assert bad not in df.columns, f'PII leaked: {bad}'
df.head()

,neighbourhood,num_listings,avg_price,median_price,avg_minimum_nights,availability_365_avg,total_reviews,reviews_per_listing,tourism_segment,priority_level
0,Centrum-West,2,195.0,195.0,2.5,100.0,43,21.5,tourist-heavy,high
1,De Baarsjes,2,135.0,135.0,3.5,130.0,48,24.0,local-heavy,low
2,De Pijp,1,135.0,135.0,2.0,210.0,44,44.0,mixed,medium
3,Oud-Noord,2,100.0,100.0,1.5,280.0,27,13.5,emerging,medium
4,Westerpark,1,155.0,155.0,2.0,140.0,36,36.0,mixed,medium


## Commit

```bash
git add .
git commit -m "HW01-A dockerized Airbnb package"
```